In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ==========================================
# 1. ЗАГРУЗКА И РАЗБИЕНИЕ ДАННЫХ
# ==========================================
# Загружаем датасет Titanic
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# Выбираем целевую переменную (Survived) и признаки (X)
X = df.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])
y = df['Survived']

# КРИТИЧЕСКИ ВАЖНО: Разбиение на train и test ДО любых трансформация
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# 2. КЛАССИФИКАЦИЯ ПРИЗНАКОВ И СТРАТЕГИИ
# ==========================================
# Числовые признаки:
# - Age: заполняем медианой (устойчива к выбросам), затем масштабируем StandardScaler.
# - Fare: заполняем медианой, затем масштабируем StandardScaler.
# - SibSp, Parch: масштабируем StandardScaler.
num_features = ['Age', 'Fare', 'SibSp', 'Parch']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Категориальные признаки:
# - Embarked, Sex, Pclass: пропуски заполняем константой 'Unknown', затем OneHotEncoder.
cat_features = ['Sex', 'Embarked', 'Pclass']

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ==========================================
# 3. СБОРКА COLUMNTRANSFORMER
# ==========================================
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

# ==========================================
# 4. ОБУЧЕНИЕ И ПРИМЕНЕНИЕ (БЕЗ УТЕЧКИ)
# ==========================================
# Обучаем трансформер ТОЛЬКО на X_train
X_train_transformed = preprocessor.fit_transform(X_train)

# Применяем (не переобучая) к X_test
X_test_transformed = preprocessor.transform(X_test)

# ==========================================
# 5. ПРОВЕРКА РЕЗУЛЬТАТОВ
# ==========================================
print("=== РЕЗУЛЬТАТЫ ВЫПОЛНЕНИЯ ПРАКТИЧЕСКОЙ РАБОТЫ №3 ===")
print(f"Размерность исходного X_train: {X_train.shape}")
print(f"Размерность исходного X_test:  {X_test.shape}\n")

print(f"Размерность преобразованного X_train: {X_train_transformed.shape}")
print(f"Размерность преобразованного X_test:  {X_test_transformed.shape}\n")

# Проверка отсутствия утечки и совпадения колонок
if X_train_transformed.shape[1] == X_test_transformed.shape[1]:
    print("Успех: Количество признаков в train и test совпадает!")
else:
    print("Ошибка: Число колонок различается!")

=== РЕЗУЛЬТАТЫ ВЫПОЛНЕНИЯ ПРАКТИЧЕСКОЙ РАБОТЫ №3 ===
Размерность исходного X_train: (712, 7)
Размерность исходного X_test:  (179, 7)

Размерность преобразованного X_train: (712, 13)
Размерность преобразованного X_test:  (179, 13)

Успех: Количество признаков в train и test совпадает!


# Практическая работа №3. Сборка пайплайна предобработки (Уровень B)

## Цель работы
Собрать пайплайн предобработки данных, исключающий утечку информации из тестовой выборки в обучающую.

## Выполненные шаги:
1. **Разбиение данных:** Данные `Titanic` разделены на train (80%) и test (20%) через `train_test_split` до применения любых транформаций для исключения Data Leakage.
2. **Обработка числовых признаков (`Age`, `Fare`, `SibSp`, `Parch`):**
   - Пропуски заполнены медианой (`SimpleImputer(strategy='median')`).
   - Применено масштабирование `StandardScaler`.
3. **Обработка категориальных признаков (`Sex`, `Embarked`, `Pclass`):**
   - Пропуски заполнены отдельной категорией `'Unknown'`.
   - Применено кодирование `OneHotEncoder(handle_unknown='ignore')`.
4. **Объединение в Pipeline:** Все шаги собраны через `ColumnTransformer`.
5. **Итоговая проверка:** Размерности обучающей и тестовой выборок после преобразования совпадают, утечка данных отсутствует.